## Packages

In [ ]:
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)   # 64-bit floats, so that results agree with numpy to ~1e-15
import matplotlib.pyplot as plt 

In [6]:
from Structure.Act_Cost import *
from Structure.DataProcess import Data_process
from Structure.NeuralNetwork import NeuralNetwork 
from Structure.Optimizer import Optimizer
from Structure.SGD import step_length, train_sgd
from Structure.PGD import train_plain
from Structure.NeuralNetRegressor import NeuralNetRegressor

## The data

In [7]:
# The data of project 1, as a starting point for part b)
np.random.seed(2026)

def runge(x):
    return 1.0 / (1.0 + 25.0 * x**2)

rng = np.random.default_rng(2026)      # fix the seed for reproducible results
n = 100 # 350 in Project 1
x = np.sort(rng.uniform(-1.0, 1.0, n))
sigma = 0.1 # 0.25 in Project 1
y = runge(x) + rng.normal(0.0, sigma, n)

# For the network the input is the raw x (one feature), samples in rows
X = x.reshape(-1, 1)
Y = y.reshape(-1, 1)  # make y a column vector for the network
#print(X.shape, Y.shape)
#plt.plot(x, runge(x), label="Runge function")
#plt.scatter(x, y, s=10, label=f"noisy data, sigma={sigma}")
#plt.legend(); plt.show()

## One hidden layer, 50 nodes with Sigmoid in hidden layer
### Crontol gradient against Jax in NeuralNetwork
### Check the Regressor against regular method for SGD (and PGD)

In [18]:
"Stochastic Gradient Descent with mini-batches"
set_epoch = 100
print("Gradient Validation, Regressor validation")
print("One hidden layer; 50 nodes")
for method in ["plain", "adam", "rmsprop"]:
    print("----------------------------")
    print(f"Training with {method}")
    # Manual Method
    network_input_size = X.shape[-1] # 9
    network_output_size = 1 # same order as y
    layer_output_sizes = [50, 1]
    activation_funcs=[sigmoid, identity]
    activation_ders=[sigmoid_der, identity_der]
    cost_fun=mse
    cost_der=mse_der

    NN = NeuralNetwork(
        network_input_size=network_input_size,
        layer_output_sizes=layer_output_sizes,
        activation_funcs=activation_funcs,
        activation_ders=activation_ders,
        cost_fun=mse,
        cost_der=mse_der,
        seed = 2026
    )

    history, last_layers = train_sgd(NN, X, Y, method=method, epochs=set_epoch, 
    batch_size=10, gamma=0.01, lmb = 0.0, penalty =None, schedule=None, seed=2026)
    MSE_SGD_M = history[-1]

    "Gradient Calculation"
    gradients = NN.compute_gradient(X, Y)
    jax_grad = NN.jax_gradient(X, Y)

    print("The gradients agree with jax.grad:", [(np.allclose(dW, dW_j), np.allclose(db, db_j))
                                    for (dW, db), (dW_j, db_j) in zip(gradients, jax_grad)])


    # Regressor
    NN_R = NeuralNetRegressor(hidden=(50,), activation="sigmoid", 
    gamma=0.01, lmbd=0.0, penalty=None, method=method, epochs=set_epoch, batch_size=10,descent = "sgd", seed=2026)
    NN_R.fit(X, Y)
    prediction_R = NN_R.predict(X)
    MSE_SGD = mse(prediction_R, Y.ravel()) # reshape to 1D for MSE calculation
            
    print(
        "SGD: The methods agree to numerical precision:",
        np.allclose(MSE_SGD, MSE_SGD_M)
    )

    "Plain Gradient Descent"
    # NeuralNetwork with plain gradient descent
    NN_p = NeuralNetwork(
        network_input_size=network_input_size,
        layer_output_sizes=layer_output_sizes,
        activation_funcs=activation_funcs,
        activation_ders=activation_ders,
        cost_fun=mse,
        cost_der=mse_der,
        seed = 2026
    )
    histroy_plain, last_layers_plain = train_plain(NN_p, X, Y, gamma=0.01, lmb = 0.0, penalty ="l2", epochs=set_epoch, seed=2026)
    mse_plain = histroy_plain[-1]

    # Regressor with plain gradient descent
    NN_R_plain = NeuralNetRegressor(hidden=(50,), activation="sigmoid",
    gamma=0.01, lmbd=0.0, penalty=None, method=method, epochs=set_epoch, batch_size=10, descent = "plain", seed=2026)
    NN_R_plain.fit(X, Y)
    prediction_R_plain = NN_R_plain.predict(X)
    mse_R_plain = mse(prediction_R_plain, Y.ravel())

    print(
        "PGD: The methods agree to numerical precision:",
        np.allclose(mse_R_plain, mse_plain)
    )  

    print(f"Train MSE SGD:{MSE_SGD:.4}", f"Train MSE PGD:{mse_R_plain:.4}")

Gradient Validation, Regressor validation
One hidden layer; 50 nodes
----------------------------
Training with plain
The gradients agree with jax.grad: [(True, True), (True, True)]
SGD: The methods agree to numerical precision: True
PGD: The methods agree to numerical precision: True
Train MSE SGD:0.1031 Train MSE PGD:0.103
----------------------------
Training with adam
The gradients agree with jax.grad: [(True, True), (True, True)]
SGD: The methods agree to numerical precision: True
PGD: The methods agree to numerical precision: True
Train MSE SGD:0.02404 Train MSE PGD:0.103
----------------------------
Training with rmsprop
The gradients agree with jax.grad: [(True, True), (True, True)]
SGD: The methods agree to numerical precision: True
PGD: The methods agree to numerical precision: True
Train MSE SGD:0.02142 Train MSE PGD:0.103


## Two hidden layer with sigmoid, 50 and 100 nodes. 


AI Declaration: This section is generated with help of Copilot, as it is a copy of the code cell above.

In [15]:
print("Gradient Validation, Regressor validation")
print("Two hidden layers; (50, 100) nodes")
for method in ["plain", "adam", "rmsprop"]:
    print("----------------------------")
    print(f"Training with {method}")
    # Manual Method
    network_input_size2 = X.shape[-1] # 9
    network_output_size2 = 1 # same order as y
    layer_output_sizes2 = [50, 100, 1]
    activation_funcs2=[sigmoid, sigmoid, identity]
    activation_ders2=[sigmoid_der, sigmoid_der, identity_der]
    cost_fun2=mse
    cost_der2=mse_der

    NN_2 = NeuralNetwork(
        network_input_size=network_input_size2,
        layer_output_sizes=layer_output_sizes2,
        activation_funcs=activation_funcs2,
        activation_ders=activation_ders2,
        cost_fun=cost_fun2,
        cost_der=cost_der2,
        seed = 2026
    )
    history_2, last_layers_2 = train_sgd(NN_2, X, Y, method=method, epochs=set_epoch, 
    batch_size=10, gamma=0.01, lmb = 0.0, penalty =None, schedule=None, seed=2026)
    MSE_SGD_M_2 = history_2[-1]


    "Gradient Calculation"
    gradients2 = NN_2.compute_gradient(X, Y)
    jax_grad2 = NN_2.jax_gradient(X, Y)

    print("The gradients agree with jax.grad:", [(np.allclose(dW, dW_j), np.allclose(db, db_j))
                                    for (dW, db), (dW_j, db_j) in zip(gradients2, jax_grad2)])

    NN_R_2 = NeuralNetRegressor(hidden=(50,100,), activation="sigmoid",
    gamma=0.01, lmbd=0.0, penalty=None, method=method, epochs=set_epoch, batch_size=10, descent = "sgd", seed=2026)
    NN_R_2.fit(X, Y)
    mse_R_2 = mse(NN_R_2.predict(X), Y.ravel())


    print(
        "SGD: The methods agree to numerical precision:",
        np.allclose(mse_R_2, MSE_SGD_M_2)
    )

    #Plain Gradient Descent with 2 hidden layers
    NN_p_2 = NeuralNetwork(
        network_input_size=network_input_size2,
        layer_output_sizes=layer_output_sizes2,
        activation_funcs=activation_funcs2,
        activation_ders=activation_ders2,
        cost_fun=cost_fun2,
        cost_der=cost_der2,
        seed = 2026
    )
    history_plain_2, last_layers_plain_2 = train_plain(NN_p_2, X, Y, gamma=0.01, lmb = 0.0, penalty ="l2", epochs=set_epoch, seed=2026)
    mse_plain_2 = history_plain_2[-1]   

    NN_R_plain_2 = NeuralNetRegressor(hidden=(50,100,), activation="sigmoid",
    gamma=0.01, lmbd=0.0, penalty=None, method=method, epochs=set_epoch, batch_size=10,descent = "plain", seed=2026)
    NN_R_plain_2.fit(X, Y)
    mse_R_plain_2 = mse(NN_R_plain_2.predict(X), Y.ravel())

    print(
        "PGD: The methods agree to numerical precision:",
        np.allclose(mse_R_plain_2, mse_plain_2)
    )
    print(f"MSE SGD:{mse_R_2:.4}", f"MSE PGD:{mse_R_plain_2:.4f}")

Gradient Validation, Regressor validation
Two hidden layers; (50, 100) nodes
----------------------------
Training with plain method
The gradients agree with jax.grad: [(True, True), (True, True), (True, True)]
SGD: The methods agree to numerical precision: True
PGD: The methods agree to numerical precision: True
MSE SGD:0.1084 MSE PGD:0.1029
----------------------------
Training with adam method
The gradients agree with jax.grad: [(True, True), (True, True), (True, True)]
SGD: The methods agree to numerical precision: True
PGD: The methods agree to numerical precision: True
MSE SGD:0.01873 MSE PGD:0.1029
----------------------------
Training with rmsprop method
The gradients agree with jax.grad: [(True, True), (True, True), (True, True)]
SGD: The methods agree to numerical precision: True
PGD: The methods agree to numerical precision: True
MSE SGD:0.01394 MSE PGD:0.1029
